# 02：质量控制与过滤

对每个来源数据集独立执行质量控制：

- **过滤前 QC 可视化**：小提琴图 + 散点图，判断数据质量分布
- **双细胞鉴定**（scrublet）：检出可能由两个细胞被误判为一个液滴产生的双重细胞
- **环境 RNA 校正**（SoupX）：扣除空液滴/碎片中的背景 RNA 污染
- **过滤**：应用基因数/线粒体比例阈值，去除低质量细胞
- **过滤后 QC 可视化**：与过滤前对比，确认阈值合理

所有步骤通过 manifest 驱动：若某数据集原作者已做某步处理（如已去双细胞），则自动跳过并记录原因，避免科学错误。

In [ ]:
# === PARAMS ===
# UPSTREAM_PATH —— 上游 01 notebook 输出（加载后从这里读取 h5ad）
# OUTPUT_PATH   —— 本 notebook 处理完成后写入的 h5ad 路径
# manifest 路径从 h5ad 元数据自动推导，无需手动指定
# QC 阈值 —— PI 可调旋钮（所有可调旋钮集中在 notebook PARAMS，方便调节）
# RSCRIPT_BIN  —— conda scrna-integration-r 环境的 Rscript 绝对路径。
#                 自动从 CONDA_PREFIX 推导同 conda 安装下的 scrna-integration-r 环境路径。
#                 如果自动定位失败或你的环境名/路径不同，直接改成绝对路径。
#                 后续 07 的 DESeq2/Monocle3/hdWGCNA 等重型 R 工具沿用同一约定。
# SOUPX_RSCRIPT —— SoupX 独立 R 脚本路径（subprocess Rscript 调用，避免 rpy2 桥接兼容性问题）

UPSTREAM_PATH = "results/01_loaded_v1.h5ad"
OUTPUT_PATH   = "results/02_qcd_v1.h5ad"

# QC 过滤阈值 —— PI 可调旋钮
#   min_genes：低于此值的细胞通常是空液滴或死细胞碎片，建议 200-500
#   max_genes：高于此值可能是双细胞（两个细胞被当成一个），建议 2500-6000
#   max_pct_mt：线粒体比例过高提示细胞膜破损/凋亡，建议 5-20
#   random_seed：随机种子，保证结果可复现
# 建议：先跑一遍看过滤前 QC 图，根据图的分布调整阈值，再跑一次
MIN_GENES     = 200
MAX_GENES     = 6000
MAX_PCT_MT    = 20
RANDOM_SEED   = 42

# R 环境配置（conda scrna-integration-r，R 4.4.3，SoupX/DESeq2/monocle3/CellChat/hdWGCNA 已装）
# RSCRIPT_BIN 通过 platform 模块统一解析（自动适配 Mac/Linux 不同 conda 路径），
# 具体赋值在下方 setup cell 中完成（确保 sys.path 先于 framework import 执行）
RSCRIPT_BIN = None  # 实际值由 setup cell 中的 platform.rscript_bin() 设置
SOUPX_RSCRIPT = "scripts/soupx_run.R"

In [ ]:
# === Setup：sys.path + 导入依赖 ===
# 确保框架 src/ 在 sys.path 上（必须在 import scrna_integration 之前）。
# 自动检测两种运行场景：从 notebooks/（Jupyter）还是项目根目录（nbconvert）启动。
import sys, os
_root = os.getcwd()
if not os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
    _root = os.path.abspath(os.path.join(_root, ".."))
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")

# 导入依赖
# scanpy：单细胞分析主库
# scipy.sparse：稀疏矩阵处理
# subprocess/shutil：SoupX 调用 Rscript 用
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
import subprocess
import shutil
import warnings
from pathlib import Path

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)

import importlib.metadata; print(f"Scanpy {importlib.metadata.version('scanpy')}  |  anndata {importlib.metadata.version('anndata')}")

# 解析 Rscript 路径（在 sys.path 就绪之后，确保 platform.py 可导入）
from scrna_integration.platform import rscript_bin
RSCRIPT_BIN = rscript_bin()
print(f"RSCRIPT_BIN: {RSCRIPT_BIN}")

In [ ]:
# 加载上游 01 输出 h5ad。
print(f"加载上游: {UPSTREAM_PATH}")
adata = sc.read_h5ad(UPSTREAM_PATH)
print(f"已加载: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")

# 识别安纳数据中含哪些来源数据集（02 必须处理多来源场景）。
source_datasets = sorted(adata.obs['source_dataset'].unique())
print(f"AnnData 中来源数据集: {source_datasets}")

In [ ]:
# 加载各来源数据集的 manifest，读取 preprocessing_done / qc_overrides。
# 02 根据 manifest 决定哪些 QC 步骤应跳过——
# 例如原作者已去除双细胞，重跑就是科学错误，而非无害重复。
manifests = {}
MANIFEST_BASE = "data"  # manifest 位于 data/{source_dataset}/manifest.yaml

for src in source_datasets:
    mf_path = Path(MANIFEST_BASE) / src.lower().replace("_", "") / "manifest.yaml"
    # 尝试常见命名模式
    candidates = [
        Path(MANIFEST_BASE) / src.lower() / "manifest.yaml",
        Path(MANIFEST_BASE) / src.lower().split("_")[0] / "manifest.yaml",
    ]
    found = None
    for c in candidates:
        if c.exists():
            found = c
            break
    # 也尝试遍历 data/ 子目录查找
    if found is None:
        for sub in Path(MANIFEST_BASE).iterdir():
            if sub.is_dir() and (sub / "manifest.yaml").exists():
                mf = yaml.safe_load((sub / "manifest.yaml").read_text())
                if mf.get("source_dataset") == src:
                    found = sub / "manifest.yaml"
                    break
    if found:
        manifests[src] = yaml.safe_load(found.read_text())
        print(f"  {src}: manifest 已从 {found} 加载")
    else:
        print(f"  {src}: manifest 未找到——假定无 QC 跳过")
        manifests[src] = {}

# 汇总各来源数据集的预处理状态。
print("\n===== 各来源数据集预处理状态 =====")
for src in source_datasets:
    mf = manifests.get(src, {})
    pp = mf.get("preprocessing_done", [])
    qc_overrides = mf.get("qc_overrides", {})
    print(f"  {src}: preprocessing_done={pp}")
    if qc_overrides:
        print(f"         qc_overrides={list(qc_overrides.keys())}")

## 过滤前 QC 可视化

绘制三个主 QC 指标的小提琴图和散点图，供 PI 在设定过滤阈值前直观判断数据质量。

**三个指标的含义**：
- `n_genes`：每个细胞检测到的基因数。过低→空液滴或死细胞；过高→可能是双细胞
- `total_counts`：每个细胞的总 UMI 计数。分布应与 n_genes 正相关
- `pct_counts_mt`：线粒体转录本百分比。过高（>20%）→细胞膜破损/凋亡

**如何使用这些图**：
1. 先看小提琴图，了解各指标的总体分布范围和离群情况
2. 再看散点图，检查 n_genes vs pct_mt 的关系——通常呈负相关，偏离主群的点可能是低质量细胞
3. 根据这些图的分布特征，回到顶部 PARAMS 调整 min_genes / max_genes / max_pct_mt

In [ ]:
# QC 可视化：按来源数据集绘制小提琴图。
# PI 据此判断各指标的分布范围，设定后续过滤阈值。
# 三个指标分别对应：检测基因数、总 UMI 计数、线粒体转录本百分比。
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, metric in enumerate(["n_genes", "total_counts", "pct_counts_mt"]):
    ax = axes[i]
    sc.pl.violin(adata, keys=metric, groupby="source_dataset",
                rotation=45, ax=ax, show=False)
    ax.set_title(f"{metric}（过滤前）")
plt.tight_layout()
fig.savefig("results/figures/02_qc_violin_pre.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# QC 可视化：散点图（n_genes vs pct_mt，total_counts vs n_genes）。
# 散点图可识别双峰分布、离群细胞簇以及线粒体比例与基因数的关系——
# 这些在小提琴图的总体分布中不易察觉。
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sc.pl.scatter(adata, x="total_counts", y="n_genes", color="pct_counts_mt",
             ax=axes[0], show=False)
axes[0].set_title("total_counts vs n_genes（按 pct_mt 着色，过滤前）")
sc.pl.scatter(adata, x="n_genes", y="pct_counts_mt", color="total_counts",
             ax=axes[1], show=False)
axes[1].set_title("n_genes vs pct_mt（按 total_counts 着色，过滤前）")
plt.tight_layout()
fig.savefig("results/figures/02_qc_scatter_pre.png", dpi=150, bbox_inches="tight")
plt.show()

## 双细胞鉴定（manifest 驱动的跳过逻辑）

按来源数据集分别进行双细胞检出：

- **跳过**：当 `preprocessing_done` 包含 `"doublet_removal"` 或
  `qc_overrides.doublet_removal.skip` 为 true。
  跳过时要求 `qc_overrides.doublet_removal.reason` 必填（manifest 规范要求）。
  被跳过的细胞写入 `obs.doublet_score = NaN` 和 `obs.predicted_doublet = False`
  （NaN 编码"不适用"，不是"数据缺失"——异构数据来源统一处理约定）。
- **运行 scrublet**：当该来源数据集不满足跳过条件时，对其子集执行 scrublet 双细胞检出。

In [ ]:
# 双细胞鉴定——按来源数据集，manifest 驱动跳过或运行。
# 每套数据可能有不同的双细胞处理历史；notebook 透明处理这种异构性。
# 列对齐：每套数据都获得相同的 obs 列，跳过者用 NaN 表示"不适用"。

# 初始化双细胞列（如不存在）。
if "doublet_score" not in adata.obs.columns:
    adata.obs["doublet_score"] = np.nan
    adata.obs["predicted_doublet"] = False

qc_doublet_skipped = {}  # 记录每来源数据集的跳过情况，供 uns["qc_skipped"] 用

for src in source_datasets:
    src_mask = adata.obs["source_dataset"] == src
    n_src = src_mask.sum()
    mf = manifests.get(src, {})
    pp_done = mf.get("preprocessing_done", [])
    qc_overrides = mf.get("qc_overrides", {})
    doublet_override = qc_overrides.get("doublet_removal", {})

    skip = False
    skip_reason = None

    if "doublet_removal" in pp_done:
        skip = True
        skip_reason = "原作者已去除双细胞（preprocessing_done 含 doublet_removal）"
    elif doublet_override.get("skip"):
        skip = True
        skip_reason = doublet_override.get("reason", "qc_overrides.doublet_removal.skip=True（无理由）")

    if skip:
        print(f"  {src} ({n_src} 细胞): 跳过 doublet_removal → {skip_reason}")
        adata.obs.loc[src_mask, "doublet_score"] = np.nan
        adata.obs.loc[src_mask, "predicted_doublet"] = False
        qc_doublet_skipped[src] = {"step": "doublet_removal", "reason": skip_reason}
    else:
        print(f"  {src} ({n_src} 细胞): 运行 scrublet...")
        sub_adata = adata[src_mask].copy()
        sc.external.pp.scrublet(sub_adata, random_state=RANDOM_SEED)
        adata.obs.loc[src_mask, "doublet_score"] = sub_adata.obs["doublet_score"].values
        adata.obs.loc[src_mask, "predicted_doublet"] = sub_adata.obs["predicted_doublet"].values
        n_doublets = int(sub_adata.obs["predicted_doublet"].sum())
        print(f"    → {n_doublets} 个预测双细胞 ({100*n_doublets/n_src:.1f}%)")
        qc_doublet_skipped[src] = {"step": "doublet_removal", "ran": True, "n_doublets": n_doublets}

adata.obs["predicted_doublet"] = adata.obs["predicted_doublet"].astype(bool)
print("\n各来源双细胞数:", adata.obs.groupby("source_dataset")["predicted_doublet"].sum().to_dict())


## 环境 RNA 校正（SoupX）——按样本 subprocess Rscript 调用

SoupX 校正环境 RNA 污染（ambient RNA contamination），利用 raw（含空液滴/碎片背景）和 filtered（只含真实细胞）两套计数矩阵的差异，估算每个基因的污染比例并扣除。

**为什么用 subprocess Rscript 而不是 rpy2？**
rpy2 + anndata2ri 在 conda R 4.4.3 下存在严重的兼容性问题：`R_GetVar` 在存有稀疏矩阵的 AnnData 上稳定崩溃，且 R 端 `SingleCellExperiment` 对象被 Python GC 提前回收导致内存越界。因此 SoupX 改为 subprocess Rscript 独立进程模式，通过临时 mtx 文件交换数据，进程隔离避免 rpy2 桥接崩溃。

**三重守卫**（任一不满足则优雅跳过，不崩溃）：
1. Manifest 声明了 `input.raw_path`
2. Rscript 可执行 + SoupX R 包可加载
3. 按样本分别处理，每个样本独立导出/调 R/读回

### SoupX 步骤一：构建输入与守卫检查

检查运行 SoupX 所需的三个前提条件——raw 矩阵是否存在、Rscript 是否可用、SoupX R 包是否可加载。任一条件不满足则优雅跳过，notebook 不崩溃。

In [ ]:
# 环境 RNA 校正（SoupX）—— subprocess Rscript 模式
#
# 工作流：对每个有 raw matrix 的样本——
#   1. 从 adata 中切出该样本的过滤后细胞，提取原始 10x barcode
#      （obs_name 格式为 "{sample_id}_{barcode}-1-{N}"，其中 -{N} 是去重后缀，
#       原始 barcode 为 "{barcode}-1"）
#   2. 导出过滤后计数矩阵到临时目录（10x mtx 格式）
#   3. subprocess 调 Rscript 跑 scripts/soupx_run.R
#   4. 读回校正后计数矩阵，**验证基因和 barcode 顺序**后写回 adata.X
#
# 守卫：三重检查（raw_path 存在 → Rscript 可用 → SoupX R 包可加载）
# 任一不满足 → 优雅跳过，不崩溃

# ---- 守卫 1：检查 raw matrix 路径 ----
raw_path = adata.uns.get("raw_matrix_path", None)

# 初始化环境校正记录列（跨来源数据集对齐）
if "ambient_correction_applied" not in adata.obs.columns:
    adata.obs['ambient_correction_applied'] = False

if raw_path is None:
    print("SoupX 已跳过: adata.uns 中无 raw_matrix_path（manifest 未声明 input.raw_path）。")
    print("  环境 RNA 校正在仅提供过滤后矩阵的数据集上物理不可行。")
else:
    print(f"raw_matrix_path 已找到: {raw_path}")

    # ---- 守卫 2：检查 Rscript 可执行 + SoupX R 包可加载 ----
    r_available = False
    _rscript = shutil.which(RSCRIPT_BIN)
    if _rscript is None:
        print(f"  Rscript 不可用: {RSCRIPT_BIN} 不存在或不在 PATH 中。")
        print(f"  请确认 conda 环境 scrna-integration-r 已创建且 Rscript 路径正确。")
    else:
        # 验证 SoupX R 包可加载
        try:
            _check = subprocess.run(
                [_rscript, "--vanilla", "-e",
                 'suppressPackageStartupMessages(library(SoupX)); cat("OK")'],
                capture_output=True, text=True, timeout=30,
            )
            if _check.returncode == 0 and "OK" in _check.stdout:
                r_available = True
                print(f"  Rscript + SoupX 可用: {_rscript}")
            else:
                print(f"  SoupX R 包加载失败 (exit={_check.returncode}):")
                print(f"  stdout: {_check.stdout.strip()}")
                print(f"  stderr: {_check.stderr.strip()[:300]}")
        except (FileNotFoundError, subprocess.TimeoutExpired) as e:
            print(f"  Rscript 调用异常: {e}")

    if not r_available:
        print("SoupX 已跳过: R 环境不可用。请确保:")
        print("  1) conda env create -f environment-r.yml")
        print("  2) conda activate scrna-integration-r")
        print('  3) 或在 R 中 install.packages("SoupX")')


### SoupX 步骤二：按样本执行校正

**导出过滤后矩阵**：从 adata 中切出该样本的过滤后细胞，提取原始 10x barcode，导出为 10x mtx 格式（基因x细胞矩阵 + barcodes.tsv + features.tsv）。

**调用 R 执行 SoupX**：通过 subprocess 调用 Rscript 执行 `scripts/soupx_run.R`，每个样本独立处理，避免跨样本污染。

**读回校正矩阵并校验**（三道防线）：
1. 形状检查——细胞数、基因数是否匹配
2. Barcode 顺序校验——逐一比对，防止静默错位
3. 基因名顺序校验——逐一比对，防止行序被 R 端重排

**写回 adata.X**：三道防线全部通过后，将校正后计数矩阵写回对应细胞的 `adata.X`。

In [ ]:
# ---- 守卫 3：按样本执行 SoupX ----
if r_available:
    raw_dir = Path(raw_path)
    soupx_script = SOUPX_RSCRIPT

    if not os.path.isfile(soupx_script):
        print(f"SoupX R 脚本不存在: {soupx_script}，跳过。")
        r_available = False

if r_available:
    # 为 SoupX 临时产出创建根目录（gitignored，与 DESeq2 _deseq2_tmp 同模式）
    soupx_tmp_root = "results/_soupx_tmp"
    os.makedirs(soupx_tmp_root, exist_ok=True)

    n_corrected = 0  # 累计校正细胞数

    for src in source_datasets:
        src_mask = adata.obs['source_dataset'] == src
        samples = sorted(adata.obs.loc[src_mask, "sample_id"].unique())
        print(f"\n  {src}: {len(samples)} 个样本")

        for sample_id in samples:
            sample_mask = src_mask & (adata.obs['sample_id'] == sample_id)
            n_cells = sample_mask.sum()
            cell_ids = adata.obs_names[sample_mask]
            print(f"    {sample_id}: {n_cells} 个细胞")

            # 提取原始 10x barcode。
            # obs_name 格式: "{sample_id}_{barcode}-1-{N}"
            # 其中 -{N}（如 -0）是跨样本去重后缀（scVI 风格），原始 barcode
            # 为去掉前缀和去重后缀后的 "{barcode}-1"（含 10x gem group 后缀）。
            # 例: "GSM7966226_ATGAGGGCAATTTCGG-1-0" → "ATGAGGGCAATTTCGG-1"
            prefix = f"{sample_id}_"
            original_barcodes = []
            for cid in cell_ids:
                if not cid.startswith(prefix):
                    warnings.warn(
                        f"cell_id '{cid}' 不以 sample prefix '{prefix}' 开头，"
                        f"整串当 barcode 用"
                    )
                    original_barcodes.append(cid)
                    continue
                rest = cid[len(prefix):]
                # 去掉末尾的 -{digit} 去重后缀
                parts = rest.rsplit('-', 1)
                if len(parts) == 2 and parts[1].isdigit():
                    raw_bc = parts[0]
                else:
                    warnings.warn(
                        f"cell_id '{cid}' barcode 解析异常"
                        f"（不匹配 {{barcode}}-{{digit}} 模式），用整串: {rest}"
                    )
                    raw_bc = rest
                original_barcodes.append(raw_bc)

            # 定位 raw_feature_bc_matrix 目录
            raw_sample_dir = raw_dir / sample_id / "raw_feature_bc_matrix"
            # 也尝试直接在 sample 目录下找
            if not (raw_sample_dir / "matrix.mtx.gz").exists() and not (raw_sample_dir / "matrix.mtx").exists():
                # fallback: raw_dir 可能就是 sample 目录本身
                alt_raw = raw_dir / "raw_feature_bc_matrix"
                if (alt_raw / "matrix.mtx.gz").exists() or (alt_raw / "matrix.mtx").exists():
                    raw_sample_dir = alt_raw
                else:
                    print(f"      -> raw_feature_bc_matrix 未找到于 {raw_sample_dir}，跳过此样本")
                    continue

            try:
                # Step 1: 导出过滤后计数矩阵（10x mtx 格式，含原始 barcodes）
                # 用此样本细胞的 adata.X（原始整数 counts）导出
                sub_adata = adata[cell_ids].copy()

                # 构造导出目录
                filtered_export = os.path.join(soupx_tmp_root, f"{sample_id}_filtered")
                os.makedirs(filtered_export, exist_ok=True)

                # 矩阵: 基因×细胞（sparse CSR → MatrixMarket）
                import scipy.io
                # SoupX R 脚本期望行=基因、列=细胞
                count_mtx = sp.csr_matrix(sub_adata.X).T  # 转置为 基因×细胞
                scipy.io.mmwrite(os.path.join(filtered_export, "matrix.mtx"), count_mtx)

                # barcodes: 使用原始 10x barcode（不含 cell_id 前缀和去重后缀）
                with open(os.path.join(filtered_export, "barcodes.tsv"), "w") as f:
                    f.write("\n".join(original_barcodes) + "\n")

                # features: 基因名（两列：gene_id + gene_name，与 10x 格式一致）
                with open(os.path.join(filtered_export, "features.tsv"), "w") as f:
                    for gn in sub_adata.var_names:
                        f.write(f"{gn}\t{gn}\tGene Expression\n")

                print(f"      过滤后矩阵已导出: {count_mtx.shape[0]} 基因 × {count_mtx.shape[1]} 细胞")

                # Step 2: subprocess 调用 Rscript 执行 SoupX
                work_dir = os.path.join(soupx_tmp_root, sample_id)
                _cmd = [
                    _rscript, "--vanilla", soupx_script,
                    os.path.abspath(work_dir),
                    os.path.abspath(filtered_export),
                    os.path.abspath(str(raw_sample_dir)),
                    sample_id,
                ]
                print(f"      执行: Rscript --vanilla {soupx_script} {sample_id}...")

                _result = subprocess.run(
                    _cmd, capture_output=True, text=True, timeout=600,
                )
                # 打印 R 脚本日志（方便调试）
                for _line in _result.stdout.strip().split("\n"):
                    print(f"        [R] {_line}")

                if _result.returncode != 0:
                    print(f"      Rscript 失败 (exit={_result.returncode}):")
                    print(f"        stderr: {_result.stderr[:500]}")
                    continue

                # Step 3: 读回校正后矩阵
                corrected_mtx_fp = os.path.join(work_dir, "corrected_counts.mtx")
                if not os.path.exists(corrected_mtx_fp):
                    print(f"      校正矩阵未产出: {corrected_mtx_fp}")
                    continue

                corrected = sp.csr_matrix(scipy.io.mmread(corrected_mtx_fp).T)
                # corrected 形状: 细胞×基因（转置回来）

                if corrected.shape != (n_cells, sub_adata.n_vars):
                    print(f"      形状不匹配: 校正矩阵 {corrected.shape} != 期望 {(n_cells, sub_adata.n_vars)}")
                    continue

                # ---- 验证基因和 barcode 顺序（防静默数据损坏） ----
                # shape 检查是第一道防线，但存在同 shape 不同顺序的风险
                # （如 R 脚本异常基因交集分支重排行序），必须逐项比对基因名和 barcode

                # 验证 barcode 顺序
                with open(os.path.join(work_dir, "barcodes.tsv")) as f:
                    out_barcodes = [line.strip() for line in f]
                if out_barcodes != original_barcodes:
                    print(f"      barcode 顺序不匹配：期望 {len(original_barcodes)}，"
                          f"得到 {len(out_barcodes)}")
                    if len(out_barcodes) > 0 and len(original_barcodes) > 0:
                        print(f"        output[0]='{out_barcodes[0]}', "
                              f"expected[0]='{original_barcodes[0]}'")
                    print(f"      跳过 {sample_id}，不写入校正矩阵")
                    continue

                # 验证基因（features）顺序
                out_features = pd.read_csv(
                    os.path.join(work_dir, "features.tsv"),
                    sep="\t", header=None,
                )
                out_genes = out_features.iloc[:, 1].tolist()
                expected_genes = sub_adata.var_names.tolist()
                if out_genes != expected_genes:
                    print(f"      基因顺序不匹配：期望 {len(expected_genes)}，"
                          f"得到 {len(out_genes)}")
                    if len(out_genes) > 0 and len(expected_genes) > 0:
                        print(f"        output[0]='{out_genes[0]}', "
                              f"expected[0]='{expected_genes[0]}'")
                    print(f"      跳过 {sample_id}，不写入校正矩阵")
                    continue

                # Step 4: 写回 adata.X（仅此样本细胞）
                adata[cell_ids].X = corrected
                adata.obs.loc[cell_ids, "ambient_correction_applied"] = True
                n_corrected += n_cells
                print(f"      SoupX 完成: {n_cells} 个细胞已校正")

                # 释放子集
                del sub_adata

            except subprocess.TimeoutExpired:
                print(f"      超时: {sample_id}（超过 10 分钟），跳过")
            except Exception as e:
                print(f"      异常 ({type(e).__name__}): {e}")
                import traceback; traceback.print_exc()


### SoupX 步骤三：校正完成报告

汇总校正状态：各样本校正细胞数、跳过原因、整体统计，并清理临时文件。

In [ ]:
print(f"\nSoupX 环境校正完成: {n_corrected} 个细胞已校正")

# 清理临时目录（防重跑积累）
shutil.rmtree(soupx_tmp_root)
print(f"临时目录已清理: {soupx_tmp_root}")

# 输出校正状态摘要
print(f"\n环境校正应用状态: {adata.obs['ambient_correction_applied'].value_counts().to_dict()}")

## 过滤

应用 PARAMS 中设定的 QC 阈值（`min_genes`、`max_genes`、`max_pct_mt`），移除低质量细胞。

**注意**：双重细胞（doublet）**不**在此阶段移除——`predicted_doublet` 标记已写入 `obs`，供下游各分析模块根据自身需求自行决定是否排除。

In [ ]:
# 过滤：应用 PARAMS 中设定的阈值。
# 各指标阈值含义：
#   min_genes=200  —— 检测到 <200 个基因的细胞通常是空液滴或死细胞碎片
#   max_genes=6000 —— 检测到 >6000 个基因的细胞可能是双重细胞（两个细胞被误判为一个液滴）
#   max_pct_mt=20  —— 线粒体转录本 >20% 提示细胞膜破损/凋亡，内容物泄漏
# 这些阈值是经验性起步值，PI 应根据过滤前 QC 可视化图调整。
print("===== QC 过滤 =====")
cells_before = adata.n_obs
print(f"过滤前细胞数: {cells_before:,}")

# 记录各指标通过/失败数，供透明审计。
passed_genes = (adata.obs['n_genes'] >= MIN_GENES) & (adata.obs['n_genes'] <= MAX_GENES)
passed_mt = adata.obs['pct_counts_mt'] <= MAX_PCT_MT
print(f"  min_genes >= {MIN_GENES}:      {(adata.obs['n_genes'] < MIN_GENES).sum():,} 失败")
print(f"  max_genes <= {MAX_GENES}:      {(adata.obs['n_genes'] > MAX_GENES).sum():,} 失败")
print(f"  基因数合并过滤:                {(~passed_genes).sum():,} 失败")
print(f"  pct_mt <= {MAX_PCT_MT}:        {(~passed_mt).sum():,} 失败")

# 使用 scanpy 原生 API 执行过滤。
sc.pp.filter_cells(adata, min_genes=MIN_GENES)
sc.pp.filter_cells(adata, max_genes=MAX_GENES)
adata = adata[adata.obs['pct_counts_mt'] <= MAX_PCT_MT, :].copy()

cells_after = adata.n_obs
print(f"\n过滤后细胞数:  {cells_after:,}")
print(f"去除细胞数:      {cells_before - cells_after:,} ({100*(cells_before-cells_after)/cells_before:.1f}%)")

## 过滤后 QC 可视化

复刻过滤前的 QC 图，供 PI 做"过滤前 vs 过滤后"直观对比。

**对比检查要点**：
- 小提琴图：各指标的分布尾部是否被正确截断（而非主体受损）
- 散点图：被移除的细胞是否集中在预期区域（低基因数 + 高线粒体比例区）
- 剩余细胞数是否合理：通常保留 80-95%，过度修剪会丢失有效数据

如果过滤后仍看到明显的离群簇，说明阈值需要收紧，回到 PARAMS 调整后重跑。

In [ ]:
# 过滤后 QC 小提琴图——与过滤前并排对比。
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, metric in enumerate(["n_genes", "total_counts", "pct_counts_mt"]):
    ax = axes[i]
    sc.pl.violin(adata, keys=metric, groupby="source_dataset",
                rotation=45, ax=ax, show=False)
    ax.set_title(f"{metric}（过滤后）")
plt.tight_layout()
fig.savefig("results/figures/02_qc_violin_post.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 过滤后 QC 散点图。
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sc.pl.scatter(adata, x="total_counts", y="n_genes", color="pct_counts_mt",
             ax=axes[0], show=False)
axes[0].set_title("total_counts vs n_genes（按 pct_mt 着色，过滤后）")
sc.pl.scatter(adata, x="n_genes", y="pct_counts_mt", color="total_counts",
             ax=axes[1], show=False)
axes[1].set_title("n_genes vs pct_mt（按 total_counts 着色，过滤后）")
plt.tight_layout()
fig.savefig("results/figures/02_qc_scatter_post.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 结构化 QC 记录——直接写 adata.uns，供下游 notebook 读取。
# qc_skipped: 按来源数据集、按步骤记录哪些步骤被跳过以及原因。
# qc_heterogeneous: 标记本次 QC 是否存在跨数据集异构（供 04 及下游读取）。
# filter_v1: 记录本次过滤运行的参数与结果，保证可复现。
import datetime

# qc_skipped —— 按来源数据集、按步骤记录
qc_skipped = {}
for src in source_datasets:
    mf = manifests.get(src, {})
    pp_done = mf.get("preprocessing_done", [])
    qc_overrides = mf.get("qc_overrides", {})
    skipped_steps = {}

    if "doublet_removal" in pp_done or qc_overrides.get("doublet_removal", {}).get("skip"):
        reason = qc_overrides.get("doublet_removal", {}).get("reason", "原作者已做预处理")
        skipped_steps["doublet_removal"] = reason

    if "basic_filter" in pp_done:
        skipped_steps["basic_filter"] = "原作者已应用基本过滤"

    if "normalization" in pp_done:
        skipped_steps["normalization"] = "原作者已做归一化"

    if skipped_steps:
        qc_skipped[src] = skipped_steps

adata.uns['qc_skipped'] = qc_skipped

# qc_heterogeneous —— 任一来数据集跳过任一步骤则为 True
adata.uns['qc_heterogeneous'] = len(qc_skipped) > 0

# filter_v1 —— 记录过滤参数与结果
adata.uns['filter_v1'] = {
    "params":     {"min_genes": MIN_GENES, "max_genes": MAX_GENES, "max_pct_mt": MAX_PCT_MT},
    "cells_in":   cells_before,
    "cells_out":  cells_after,
    "timestamp":  datetime.datetime.now().isoformat(),
}

print("qc_skipped:", qc_skipped)
print("qc_heterogeneous:", adata.uns['qc_heterogeneous'])
print("filter_v1:", adata.uns['filter_v1'])

In [ ]:
# 内存安全自检（写入前单行断言）。
# 守护最高风险的内存退化：adata.X 变为稠密或丢失 float32。
# 如断言失败，追溯是哪步操作 densify 或 cast 了矩阵。
import scipy.sparse as sp
import numpy as np
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, (
    f"adata.X 不变量被破坏: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"
)
print("内存自检通过: X 是 sparse CSR float32。")

In [ ]:
# 写入本 stage checkpoint 到磁盘。
# compression="lzf"——比 gzip 快，
# 比未压缩约省 30% 空间，且保留 sparse CSR 布局。
adata.write_h5ad(OUTPUT_PATH, compression="lzf")
print(f"已写出 {OUTPUT_PATH}")

# 校验文件已写入且可读。
import os
assert os.path.exists(OUTPUT_PATH), f"输出未找到: {OUTPUT_PATH}"
print(f"已校验: {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH):,} bytes)")

In [ ]:
# 跨 notebook 边界释放内存。
# 没有这段的话，Jupyter 内核会保留上一 stage 的 AnnData，
# 在同一内核会话中运行下一 stage 时造成内存叠加——
# 跨 7 个 notebook × 多个版本，在 25k 细胞数据上可撑爆 16GB 内存。
del adata
import gc
gc.collect()
print("内存已释放。")